# **COLAB CONFIGURATION**

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
%cd /content/drive/MyDrive/College/FLEX-Med/flex-med

/content/drive/MyDrive/College/FLEX-Med/flex-med


# **DATASET SETUP**
- Set up the public dataset with 20% CNMC and 20% ALL_IDB2
- New ALLIDB and CNMC dataset

In [ ]:
import os
import shutil
import random
import glob
from pathlib import Path
from tqdm import tqdm

# <---------------- CONFIGURATION ---------------->
# Define where your uploaded data sits currently
SOURCE_ROOT = "/content/drive/MyDrive/College/Datasets"
PATH_ALLIDB = os.path.join(SOURCE_ROOT, "ALL_IDB2", "img")
PATH_CNMC_ALL = os.path.join(SOURCE_ROOT, "CNMC", "all")
PATH_CNMC_HEM = os.path.join(SOURCE_ROOT, "CNMC", "hem")

# Define where you want the new Federation-ready data to go
OUTPUT_ROOT = "/content/drive/MyDrive/College/Datasets/fed_data"
SPLIT_RATIO = 0.2  # 20% for Public Anchor

# <---------------- SETUP DIRECTORIES ---------------->
def create_dir_structure(base_path):
    for split in ["public_anchor", "client_cnmc", "client_allidb"]:
        for label in ["all", "hem"]:
            os.makedirs(os.path.join(base_path, split, label), exist_ok=True)

print(f"Creating directory structure at {OUTPUT_ROOT}...")
create_dir_structure(OUTPUT_ROOT)

# <---------------- HELPER FUNCTIONS ---------------->
def copy_files(file_list, destination_split, label, dataset_prefix):
    """
    Copies files to destination.
    Renames them with prefix to avoid collisions in the Public folder.
    """
    dest_dir = os.path.join(OUTPUT_ROOT, destination_split, label)

    for src_path in file_list:
        filename = os.path.basename(src_path)
        new_filename = f"{dataset_prefix}_{filename}"
        dst_path = os.path.join(dest_dir, new_filename)
        shutil.copy2(src_path, dst_path)

# <---------------- PROCESS ALL-IDB2 ---------------->
print("\nProcessing ALL-IDB2...")
allidb_images = glob.glob(os.path.join(PATH_ALLIDB, "*"))
allidb_all = [] # Blasts (1)
allidb_hem = [] # Healthy (0)

# Parse filenames based on: ImXXX_Y.jpg (Y=0 Healthy, Y=1 ALL)
for img_path in allidb_images:
    filename = os.path.basename(img_path)
    # Extract the digit before .jpg
    try:
        label_char = filename.split('_')[-1].split('.')[0]
        if label_char == '1':
            allidb_all.append(img_path)
        elif label_char == '0':
            allidb_hem.append(img_path)
    except Exception as e:
        print(f"Skipping malformed file: {filename}")

# Shuffle
random.shuffle(allidb_all)
random.shuffle(allidb_hem)

# Split ALL (Leukemia)
split_idx_all = int(len(allidb_all) * SPLIT_RATIO)
public_allidb_all = allidb_all[:split_idx_all]
private_allidb_all = allidb_all[split_idx_all:]

# Split HEM (Healthy)
split_idx_hem = int(len(allidb_hem) * SPLIT_RATIO)
public_allidb_hem = allidb_hem[:split_idx_hem]
private_allidb_hem = allidb_hem[split_idx_hem:]

# Execute Copy
# 1. To Public Anchor
copy_files(public_allidb_all, "public_anchor", "all", "allidb")
copy_files(public_allidb_hem, "public_anchor", "hem", "allidb")

# 2. To Client Private (Client 0)
copy_files(private_allidb_all, "client_allidb", "all", "allidb")
copy_files(private_allidb_hem, "client_allidb", "hem", "allidb")

print(f"ALL-IDB2 Done. Public: {len(public_allidb_all)+len(public_allidb_hem)}, Private: {len(private_allidb_all)+len(private_allidb_hem)}")

# <---------------- PROCESS CNMC ---------------->
print("\nProcessing CNMC...")

# Get file lists
cnmc_all = glob.glob(os.path.join(PATH_CNMC_ALL, "*"))
cnmc_hem = glob.glob(os.path.join(PATH_CNMC_HEM, "*"))

# Shuffle
random.shuffle(cnmc_all)
random.shuffle(cnmc_hem)

# Split ALL
split_idx_all = int(len(cnmc_all) * SPLIT_RATIO)
public_cnmc_all = cnmc_all[:split_idx_all]
private_cnmc_all = cnmc_all[split_idx_all:]

# Split HEM
split_idx_hem = int(len(cnmc_hem) * SPLIT_RATIO)
public_cnmc_hem = cnmc_hem[:split_idx_hem]
private_cnmc_hem = cnmc_hem[split_idx_hem:]

# Execute Copy
# 1. To Public Anchor
copy_files(public_cnmc_all, "public_anchor", "all", "cnmc")
copy_files(public_cnmc_hem, "public_anchor", "hem", "cnmc")

# 2. To Client Private (Client 1)
copy_files(private_cnmc_all, "client_cnmc", "all", "cnmc")
copy_files(private_cnmc_hem, "client_cnmc", "hem", "cnmc")

print(f"CNMC Done. Public: {len(public_cnmc_all)+len(public_cnmc_hem)}, Private: {len(private_cnmc_all)+len(private_cnmc_hem)}")

# <---------------- FINAL SUMMARY ---------------->
print("\n" + "="*30)
print("DATASET GENERATION COMPLETE")
print("="*30)
print(f"New Public Anchor Size (Total): {len(os.listdir(os.path.join(OUTPUT_ROOT, 'public_anchor', 'all'))) + len(os.listdir(os.path.join(OUTPUT_ROOT, 'public_anchor', 'hem')))}")
print(f"Client ALL-IDB2 Size (Private): {len(os.listdir(os.path.join(OUTPUT_ROOT, 'client_allidb', 'all'))) + len(os.listdir(os.path.join(OUTPUT_ROOT, 'client_allidb', 'hem')))}")
print(f"Client CNMC Size (Private):     {len(os.listdir(os.path.join(OUTPUT_ROOT, 'client_cnmc', 'all'))) + len(os.listdir(os.path.join(OUTPUT_ROOT, 'client_cnmc', 'hem')))}")
print(f"\nData is ready at: {OUTPUT_ROOT}")

# **FLOWER RUN**

In [ ]:
!python -m pip install flwr

In [ ]:
!python -m pip install "flwr[simulation]"

In [ ]:
!pip install -e .

Obtaining file:///content/drive/MyDrive/College/FLEX-Med/flex-med
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for flex-med (pyproject.toml) ... done
  Created wheel for flex-med: filename=flex_med-1.0.0-py2.py3-none-any.whl size=1025 sha256=380b8e2f07fefbc033fe0f17bcde24904fad927eb4f737acf188d26fd9405d03
  Stored in directory: /tmp/pip-ephem-wheel-cache-26_l2vbm/wheels/6f/12/a8/e6efada7cedbb08b2d6ad6d9a40391396f3a38b80068ac38f2
Successfully built flex-med
  Attempting uninstall: flex-med
    Found existing installation: flex-med 1.0.0
    Uninstalling flex-med-1.0.0:
      Successfully uninstalled flex-med-1.0.0


In [ ]:
!flwr run .

Loading project configuration... 
Success
2025-12-07 13:16:32.357632: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-07 13:16:32.384501: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765113392.411018   45459 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765113392.417985   45459 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765113392.441801   45459 computation_placer.cc:177] computation placer already registere

# **Inference code where we can pass the image path**

In [ ]:
import sys
import os

# ==========================================
# 0. SETUP PATHS
# ==========================================
PACKAGE_ROOT = "/content/drive/MyDrive/College/FLEX-Med/flex-med/"
if PACKAGE_ROOT not in sys.path:
    sys.path.append(PACKAGE_ROOT)

import torch
import matplotlib.pyplot as plt
import numpy as np
import re
from PIL import Image
from torchvision import datasets

from flex_med.task import (
    get_model_by_type,
    COMMON_TRANSFORM,
    IMG_SIZE
)

# ==========================================
# 1. CONFIGURATION
# ==========================================

MODEL_PATH = "/content/drive/MyDrive/College/models/model_client_0.pt"   # Change for other clients
TEST_DATA_PATH = "/content/drive/MyDrive/College/Datasets/fed_data/public_anchor"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASSES = ['ALL (Leukemia)', 'Hem (Healthy)']

# ==========================================
# 2. AUTOMATIC ARCHITECTURE SELECTOR
# ==========================================
def get_model_and_name(path):
    match = re.search(r'model_client_(\d+).pt', path)

    if not match:
        print("Could not deduce Client ID from filename. Defaulting to ResNet.")
        return get_model_by_type('resnet18'), "ResNet18 (Default)"

    client_id = int(match.group(1))
    mod = client_id % 3

    if mod == 0:
       return get_model_by_type('resnet18'), f"ResNet18 (Client {client_id} - ALL-IDB2)"
    elif mod == 1:
       return get_model_by_type('mobilenet_v2'), f"MobileNetV2 (Client {client_id} - CNMC)"
    else:
       return get_model_by_type('efficientnet_b3'), f"EfficientNet-B3 (Client {client_id} - Free Rider)"

# ==========================================
# 3. LOAD MODEL
# ==========================================
if not os.path.exists(MODEL_PATH):
    print(f"ERROR: Model file not found at: {MODEL_PATH}")
    exit()

print(f"Loading model from: {MODEL_PATH}")
model, model_name = get_model_and_name(MODEL_PATH)
model.to(DEVICE)

try:
    state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    print(f"✓ Successfully loaded: {model_name}")
except Exception as e:
    print(f"Error loading weights: {e}")
    exit()

# ==========================================
# 4. LOAD TEST DATA (Optional for random)
# ==========================================
print(f"Loading Test Data from: {TEST_DATA_PATH}")
if not os.path.exists(TEST_DATA_PATH):
    print(f"Error: Data path {TEST_DATA_PATH} does not exist.")
    exit()

ds_test = datasets.ImageFolder(root=TEST_DATA_PATH, transform=COMMON_TRANSFORM)
print(f"Found {len(ds_test)} images.")

# ==========================================
# 5. VISUALIZATION HELPER
# ==========================================
def unnormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return tensor * std + mean

# ==========================================
# 6. RANDOM PREDICTION FUNCTION
# ==========================================
def predict_random_image():
    index = np.random.randint(len(ds_test))
    image_tensor, true_label_idx = ds_test[index]
    true_label_name = CLASSES[true_label_idx]

    input_batch = image_tensor.unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(input_batch)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        predicted_idx = outputs.argmax(dim=1).item()
        confidence = probabilities[0][predicted_idx].item()

    predicted_name = CLASSES[predicted_idx]
    color = 'green' if predicted_idx == true_label_idx else 'red'

    disp_img = unnormalize(image_tensor.cpu()).permute(1, 2, 0).numpy()
    disp_img = np.clip(disp_img, 0, 1)

    title_text = (
        f"Model: {model_name}\n"
        f"Actual: {true_label_name}\n"
        f"Pred: {predicted_name} ({confidence:.1%})"
    )

    plt.figure(figsize=(5, 5))
    plt.imshow(disp_img)
    plt.title(title_text, color=color, fontweight='bold', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# ==========================================
# 7. MANUAL IMAGE PREDICTION FUNCTION
# ==========================================
def predict_image_from_path(img_path):
    if not os.path.exists(img_path):
        print(f"Error: Image path not found: {img_path}")
        return

    # ----------------------------------
    # 1. Infer TRUE LABEL from folder name
    # ----------------------------------
    parent_folder = os.path.basename(os.path.dirname(img_path)).lower()

    label_map = {
        "all": 0,   # ALL (Leukemia)
        "hem": 1    # Healthy
    }

    true_label_idx = label_map.get(parent_folder, None)
    true_label_name = CLASSES[true_label_idx] if true_label_idx is not None else "Unknown"

    # ----------------------------------
    # 2. Load & Transform Image
    # ----------------------------------
    img = Image.open(img_path).convert("RGB")
    img_tensor = COMMON_TRANSFORM(img)
    input_batch = img_tensor.unsqueeze(0).to(DEVICE)

    # ----------------------------------
    # 3. Model Prediction
    # ----------------------------------
    with torch.no_grad():
        outputs = model(input_batch)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        predicted_idx = outputs.argmax(dim=1).item()
        confidence = probabilities[0][predicted_idx].item()

    predicted_name = CLASSES[predicted_idx]

    # ----------------------------------
    # 4. Correctness + Color
    # ----------------------------------
    is_correct = (predicted_idx == true_label_idx)
    color = "green" if is_correct else "red"
    verdict = "✓ Correct" if is_correct else "✗ Incorrect"

    # ----------------------------------
    # 5. Display Image
    # ----------------------------------
    disp_img = unnormalize(img_tensor.cpu()).permute(1, 2, 0).numpy()
    disp_img = np.clip(disp_img, 0, 1)

    title_text = (
        f"Model: {model_name}\n"
        f"Actual: {true_label_name}\n"
        f"Predicted: {predicted_name} ({confidence:.1%})\n"
        f"{verdict}"
    )

    plt.figure(figsize=(5, 5))
    plt.imshow(disp_img)
    plt.title(title_text, color=color, fontweight="bold", fontsize=12)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# ==========================================
# 8. RUN
# ==========================================

# Example 1: Random predictions from dataset
for i in range(3):
    predict_random_image()

# Example 2: Manual image path prediction
# predict_image_from_path("/content/drive/MyDrive/College/Datasets/fed_data/public_anchor/hem/allidb_Im132_0.tif")

# **SSH SETUP**

In [ ]:
# Run Once
!mkdir -p /content/drive/MyDrive/colab_ssh_keys

In [ ]:
# Run Once
!ssh-keygen -t ed25519 -C "s.shrenikdeep@gmail.com" -f /content/drive/MyDrive/colab_ssh_keys/colab_key -N ""

Generating public/private ed25519 key pair.
Your identification has been saved in /content/drive/MyDrive/colab_ssh_keys/colab_key
Your public key has been saved in /content/drive/MyDrive/colab_ssh_keys/colab_key.pub
The key fingerprint is:
SHA256:ffV8lP73yUIY24qx3KtmShD18R7atyh0wY/fmG+XrP4 s.shrenikdeep@gmail.com
The key's randomart image is:
+--[ED25519 256]--+
|       . .       |
|      . . +     .|
|     .   . =  ...|
|      .  .+.=..+ |
|     .  So.=*o .+|
|      . ...++o= o|
|       ...=.o=..+|
|      .  *.o .o+=|
|       .+...oo=Eo|
+----[SHA256]-----+


In [ ]:
# Run Once
!cat /content/drive/MyDrive/colab_ssh_keys/colab_key.pub

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIOj8H7Axo8HqpWKpdU9ZLXH5zcoreXgp/CrZygcbq3Qy s.shrenikdeep@gmail.com


# **GITHUB SETUP**

In [8]:
!git config --global user.name "Shrenik Deep"
!git config --global user.email "s.shrenikdeep@gmail.com"

In [9]:
!mkdir -p ~/.ssh

# Copy key from Drive to the VM
!cp /content/drive/MyDrive/colab_ssh_keys/colab_key ~/.ssh/id_ed25519
!cp /content/drive/MyDrive/colab_ssh_keys/colab_key.pub ~/.ssh/id_ed25519.pub

# Secure permissions
!chmod 600 ~/.ssh/id_ed25519

# Add GitHub to known hosts
!ssh-keyscan -H github.com >> ~/.ssh/known_hosts

# github.com:22 SSH-2.0-1e748d5
# github.com:22 SSH-2.0-1e748d5
# github.com:22 SSH-2.0-1e748d5
# github.com:22 SSH-2.0-1e748d5
# github.com:22 SSH-2.0-1e748d5


In [10]:
!ssh -T git@github.com

Hi Shrenik0321! You've successfully authenticated, but GitHub does not provide shell access.


In [11]:
!git add .
!git commit -m "updates from colab"
!git push

[main d995d0e] updates from colab
 5 files changed, 387 insertions(+), 356 deletions(-)
 rewrite flex-med/final_consensus.npy (98%)
 rewrite flex-med/flex_med/client_app.py (64%)
 create mode 100644 flex-med/flex_med/db_config.py
 rewrite flex-med/pyproject.toml (63%)
Enumerating objects: 16, done.
Counting objects: 100% (16/16), done.
Delta compression using up to 2 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 20.47 KiB | 635.00 KiB/s, done.
Total 9 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To github.com:Shrenik0321/FLEX-Med.git
   148601e..d995d0e  main -> main


In [ ]:
!pip install supabase

# **Evaluation Metrics**

In [ ]:
# ==========================================
# FULL TEST DATASET EVALUATION (SINGLE CELL)
# ==========================================

import sys, os, re
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import datasets
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# ==========================================
# PATH SETUP
# ==========================================
PACKAGE_ROOT = "/content/drive/MyDrive/College/FLEX-Med/flex-med/"
if PACKAGE_ROOT not in sys.path:
    sys.path.append(PACKAGE_ROOT)

from flex_med.task import (
    get_model_by_type,
    COMMON_TRANSFORM
)

# ==========================================
# CONFIG
# ==========================================
MODEL_PATH = "/content/drive/MyDrive/College/models/model_client_0.pt"
TEST_DATA_PATH = "/content/drive/MyDrive/College/Datasets/fed_data/public_anchor"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ['ALL (Leukemia)', 'Hem (Healthy)']

# ==========================================
# MODEL SELECTION (AUTO)
# ==========================================
def get_model_and_name(path):
    match = re.search(r'model_client_(\d+).pt', path)
    client_id = int(match.group(1)) if match else 0

    if client_id % 3 == 0:
        return get_model_by_type('resnet18'), f"ResNet18 (Client {client_id})"
    elif client_id % 3 == 1:
        return get_model_by_type('mobilenet_v2'), f"MobileNetV2 (Client {client_id})"
    else:
        return get_model_by_type('efficientnet_b3'), f"EfficientNet-B3 (Client {client_id})"

# ==========================================
# LOAD MODEL
# ==========================================
assert os.path.exists(MODEL_PATH), "Model file not found"

model, model_name = get_model_and_name(MODEL_PATH)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

print(f"✓ Loaded model: {model_name}")

# ==========================================
# LOAD DATA
# ==========================================
assert os.path.exists(TEST_DATA_PATH), "Test data path not found"
ds_test = datasets.ImageFolder(TEST_DATA_PATH, transform=COMMON_TRANSFORM)
print(f"✓ Loaded {len(ds_test)} test images")

# ==========================================
# EVALUATION
# ==========================================
y_true, y_pred = [], []

with torch.no_grad():
    for img, label in ds_test:
        img = img.unsqueeze(0).to(DEVICE)
        output = model(img)
        pred = output.argmax(dim=1).item()

        y_true.append(label)
        y_pred.append(pred)

# ==========================================
# METRICS
# ==========================================
acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="binary")
rec  = recall_score(y_true, y_pred, average="binary")
f1   = f1_score(y_true, y_pred, average="binary")
cm   = confusion_matrix(y_true, y_pred)

print("\n📊 Evaluation Metrics")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

# ==========================================
# CONFUSION MATRIX
# ==========================================
plt.figure(figsize=(4, 4))
plt.imshow(cm, cmap="Blues")
plt.title(f"Confusion Matrix\n{model_name}", fontweight="bold")
plt.xticks([0, 1], CLASSES)
plt.yticks([0, 1], CLASSES)

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=12)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

# ==========================================
# METRIC BAR CHART
# ==========================================
metrics = [acc, prec, rec, f1]
labels = ["Accuracy", "Precision", "Recall", "F1-score"]

plt.figure(figsize=(6, 4))
bars = plt.bar(labels, metrics)
plt.ylim(0, 1)
plt.title(f"Model Performance\n{model_name}", fontweight="bold")

for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.02,
             f"{bar.get_height():.2f}",
             ha="center")

plt.tight_layout()
plt.show()

# **Dynamically Start Federated Learning**

In [3]:
# Install dependencies needed for FastAPI, Uvicorn, ngrok, and async patching
!pip install fastapi uvicorn pyngrok nest-asyncio -q

# --- Imports ---
import nest_asyncio
import uvicorn
import subprocess
import threading
from pyngrok import ngrok
from pyngrok.process import kill_process
from fastapi import FastAPI, BackgroundTasks, HTTPException
from starlette.middleware.cors import CORSMiddleware

In [4]:
import os
import sys
import io
import logging
import traceback
import subprocess
import nest_asyncio
import asyncio
from fastapi import FastAPI, BackgroundTasks, HTTPException, File, UploadFile
from starlette.middleware.cors import CORSMiddleware
from starlette.responses import JSONResponse
from PIL import Image
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms

# ==========================================
# 1. CONFIGURATION & LOGGING
# ==========================================

# Define Paths & Globals
MODEL_PATH = "/content/drive/MyDrive/College/models/model_client_0.pt"
LOG_DIR = "/content/drive/MyDrive/College/FLEX-Med/flex-med"
CLASSES = ["ALL", "HEM"]  # <--- Verify these match your training classes
MODEL_NAME = "ResNet18"   # Used for response data

# Setup Logging
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE_PATH = os.path.join(LOG_DIR, "inference_debug.log")

logger = logging.getLogger("inference_logger")
logger.setLevel(logging.INFO)

# Clear existing handlers to prevent duplicates on re-run
if logger.hasHandlers():
    logger.handlers.clear()

# Add Handlers
file_handler = logging.FileHandler(LOG_FILE_PATH)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))

logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.info("=== Server Setup Started - Logging Initialized ===")

# ==========================================
# 2. MODEL LOADING & TRANSFORMS
# ==========================================

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Define Transformations
COMMON_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Initialize Model Variable
model = None

logger.info(f"Attempting to load model from {MODEL_PATH}...")

try:
    if os.path.exists(MODEL_PATH):
        # A) Initialize Architecture (ResNet18)
        model = models.resnet18(pretrained=False)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, len(CLASSES))

        # B) Load Weights
        state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
        model.load_state_dict(state_dict)

        # C) Move to Device
        model = model.to(DEVICE)
        model.eval() # Set to evaluation mode

        logger.info(f"✅ Model loaded successfully on {DEVICE}!")
    else:
        logger.error(f"❌ Model file not found at: {MODEL_PATH}")
        print(f"WARNING: File {MODEL_PATH} does not exist. Inference will fail.")

except Exception as e:
    logger.error(f"❌ Error loading model: {e}")
    logger.error(traceback.format_exc())
    print("CRITICAL: Model failed to load. Check logs.")

# ==========================================
# 3. BACKGROUND TASKS (FLOWER)
# ==========================================

def run_flower_server_in_colab():
    command_sequence = (
        "echo '--- Starting FL Setup ---' && "
        "cd /content/drive/MyDrive/College/FLEX-Med/flex-med && "
        "echo 'Current Directory:' && pwd && "
        "python -m pip install flwr 'flwr[simulation]' -q && "
        "pip install -e . -q && "
        "echo '--- Starting flwr run ---' && "
        "flwr run ."
    )
    logger.info("[FL-ORCHESTRATOR] Starting background Flower job...")
    try:
        result = subprocess.run(
            command_sequence, shell=True, capture_output=True, text=True, check=True
        )
        logger.info(f"[FL-ORCHESTRATOR] Job Finished. Output:\n{result.stdout}")
    except subprocess.CalledProcessError as e:
        logger.error(f"[FL-ORCHESTRATOR] Job Failed with code {e.returncode}. Stderr:\n{e.stderr}")
    except Exception as e:
        logger.error(f"[FL-ORCHESTRATOR] General Error: {e}")

# ==========================================
# 4. FASTAPI APP & ENDPOINTS
# ==========================================

app = FastAPI(title="Flower FL Orchestrator")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def read_root():
    return {"message": "Server is operational. Model status: " + ("Loaded" if model else "Not Loaded")}

@app.post("/inference")
async def run_inference(file: UploadFile = File(...)):
    # Wrap ENTIRE function in try-except to catch initialization errors
    try:
        logger.info(f"Received inference request for: {file.filename}")

        # 1. Check Model Availability
        if model is None:
            logger.critical("Inference failed: Model variable is None.")
            raise HTTPException(status_code=503, detail="Model not loaded on server.")

        # 2. Read & Process Image
        try:
            image_bytes = await file.read()
            image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            logger.info(f"Image processed. Size: {image.size}")
        except Exception as e:
            logger.error(f"Image processing failed: {e}")
            raise HTTPException(status_code=400, detail="Invalid image file.")

        # 3. Transform
        try:
            img_tensor = COMMON_TRANSFORM(image)
            input_batch = img_tensor.unsqueeze(0).to(DEVICE)
        except Exception as e:
            logger.error(f"Transform failed: {e}")
            raise HTTPException(status_code=500, detail="Image transformation failed.")

        # 4. Prediction
        with torch.no_grad():
            outputs = model(input_batch)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            predicted_idx = outputs.argmax(dim=1).item()
            confidence = probabilities[0][predicted_idx].item()

        predicted_name = CLASSES[predicted_idx]
        logger.info(f"Result: {predicted_name} ({confidence:.4f})")

        return JSONResponse(
            status_code=200,
            content={
                "prediction": predicted_name,
                "confidence": float(confidence),
                "class": int(predicted_idx),
                "model": MODEL_NAME,
                "all_probabilities": {
                    CLASSES[i]: float(probabilities[0][i].item())
                    for i in range(len(CLASSES))
                }
            }
        )

    except HTTPException as he:
        raise he # Pass through standard HTTP errors
    except Exception as e:
        # Catch unforeseen errors (variables missing, logic bugs)
        error_msg = f"Unexpected Server Error: {str(e)}"
        logger.error(error_msg, exc_info=True)
        raise HTTPException(status_code=500, detail=error_msg)

@app.post("/start_fl")
async def start_fl(background_tasks: BackgroundTasks):
    background_tasks.add_task(run_flower_server_in_colab)
    return JSONResponse(status_code=202, content={"status": "accepted", "message": "Flower job queued."})

print("\n✅ Setup Complete. Run your Ngrok/Server cell now.")

INFO: === Server Setup Started - Logging Initialized ===


INFO:inference_logger:=== Server Setup Started - Logging Initialized ===


INFO: Attempting to load model from /content/drive/MyDrive/College/models/model_client_0.pt...


INFO:inference_logger:Attempting to load model from /content/drive/MyDrive/College/models/model_client_0.pt...
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


INFO: ✅ Model loaded successfully on cpu!


INFO:inference_logger:✅ Model loaded successfully on cpu!



✅ Setup Complete. Run your Ngrok/Server cell now.


In [5]:
# 0. Cleanup from previous runs
try:
    ngrok.disconnect()
    kill_process()
except Exception:
    pass # Ignore errors if no process was running

# 1. Configure and Start ngrok (Replace with your token)
AUTH_TOKEN = "2nI3IUybQHGO30McAH7x4UJRiND_695D6z3DbKjWLrq7N1ATy"
ngrok.set_auth_token(AUTH_TOKEN)

# Start the ngrok tunnel on the FastAPI port (8000)
http_tunnel = ngrok.connect(8000)
public_url = http_tunnel.public_url

print(f"🌐 Public API URL: {public_url}")
print(f"💡 Test Endpoint: {public_url}/")
print(f"🚀 FL Trigger: {public_url}/start_fl")


# 2. Apply Colab Fix and Run Uvicorn in Thread
nest_asyncio.apply()

# Function to run uvicorn
def run_uvicorn():
    # host="0.0.0.0" ensures it's available to ngrok
    uvicorn.run(app, port=8000, host="0.0.0.0")

# Start the FastAPI server in a new thread
thread = threading.Thread(target=run_uvicorn)
thread.start()

print("\nFastAPI server started in a background thread.")

🌐 Public API URL: https://f7e41b172747.ngrok-free.app
💡 Test Endpoint: https://f7e41b172747.ngrok-free.app/
🚀 FL Trigger: https://f7e41b172747.ngrok-free.app/start_fl

FastAPI server started in a background thread.


In [ ]:
# Kill any process running on port 8000
!fuser -k 8000/tcp

print("Attempted to kill processes on port 8000. Please re-run the FastAPI server cell now.")